<!-- NB_DOC_INTRO_V1 -->
# ML — Bagging, Boosting, Random Forest, GBM

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Methodes d'**ensemble** = combiner plusieurs modeles faibles pour faire mieux que chacun. Deux familles : **Bagging** (Bootstrap Aggregating — RF) reduit la variance, **Boosting** (AdaBoost, GBM) reduit le biais. Ce notebook explique le **trade-off biais/variance** + implemente et compare les algorithmes.

## Plan

1. Setup + decomposition biais/variance
2. Bagging (BaggingClassifier, Random Forest, Extra-Trees)
3. Boosting (AdaBoost, GradientBoosting, HistGradientBoosting)
4. XGBoost / LightGBM / CatBoost (presentation)
5. Comparatif bench
6. Stacking et Voting
7. Pieges et anti-patterns
8. References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier,
    AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier,
    VotingClassifier, StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
import time
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Decomposition biais/variance

L'erreur d'un modele se decompose en : **Biais²** + **Variance** + **Bruit irreductible**.

- **Biais eleve** : modele trop simple (underfitting) — ex: regression lineaire sur donnees non-lineaires
- **Variance elevee** : modele trop complexe (overfitting) — ex: arbre sans `max_depth`

**Bagging** ↓ variance (vote/moyenne sur modeles entraines sur bootstrap samples).
**Boosting** ↓ biais (modeles sequentiels qui corrigent les erreurs precedentes).

In [ ]:
# Demo visuelle : trade-off biais/variance avec decision trees a profondeur variable
X, y = make_moons(n_samples=300, noise=0.3, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED)

depths = [1, 3, 10, None]   # None = sans limite (overfit)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))

for ax, d in zip(axes, depths):
    clf = DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(X_tr, y_tr)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", edgecolors="k", s=30)
    ax.set_title(f"max_depth={d}\nTrain acc {clf.score(X_tr, y_tr):.2f} | Test acc {clf.score(X_te, y_te):.2f}")
plt.tight_layout(); plt.show()
print("Note : max_depth=1 -> underfit (haut biais), max_depth=None -> overfit (haute variance)")

## 2. Bagging — BaggingClassifier, Random Forest, Extra-Trees

In [ ]:
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15,
                            n_redundant=2, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

# 1. BaggingClassifier (generique : prend n'importe quel estimateur)
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50, max_samples=0.8,    # 80% du train par estimateur (bootstrap)
    bootstrap=True, random_state=SEED, n_jobs=-1,
)
bag.fit(X_tr, y_tr)
print(f"Bagging  : {bag.score(X_te, y_te):.4f}")

# 2. RandomForest : Bagging + sub-sampling de features
rf = RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=SEED, n_jobs=-1)
rf.fit(X_tr, y_tr)
print(f"RandomForest : {rf.score(X_te, y_te):.4f}")

# 3. Extra-Trees : RF + splits aleatoires (encore plus de variance reduction)
et = ExtraTreesClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
et.fit(X_tr, y_tr)
print(f"ExtraTrees   : {et.score(X_te, y_te):.4f}")

**Quand utiliser quoi ?**
- **BaggingClassifier** : si tu veux bagger un autre modele (SVM, KNN, ...)
- **RandomForest** : 90% des cas — robuste, peu de tuning requis
- **ExtraTrees** : datasets avec peu de bruit, decisions tres aleatoires

## 3. Boosting — AdaBoost, GradientBoosting, HistGradientBoosting

In [ ]:
# AdaBoost : reweight les erreurs
ada = AdaBoostClassifier(n_estimators=100, random_state=SEED)
ada.fit(X_tr, y_tr)
print(f"AdaBoost              : {ada.score(X_te, y_te):.4f}")

# Gradient Boosting : fit sur les residus (gradient de la loss)
gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=SEED)
gbm.fit(X_tr, y_tr)
print(f"GradientBoosting      : {gbm.score(X_te, y_te):.4f}")

# HistGradientBoosting : equivalent LightGBM (binning + tres rapide + supporte NaN)
hgbm = HistGradientBoostingClassifier(max_iter=100, max_depth=5, random_state=SEED)
hgbm.fit(X_tr, y_tr)
print(f"HistGradientBoosting  : {hgbm.score(X_te, y_te):.4f}")

## 4. XGBoost / LightGBM / CatBoost — les rois du tabular 2025

| Lib | Forces | Cas |
|---|---|---|
| **XGBoost** | Le plus mature, doc enorme, GPU support | Kaggle classique, valeur sure |
| **LightGBM** | Tres rapide (leaf-wise), low memory | Gros datasets, prod |
| **CatBoost** | Gere natif categorielles, peu de tuning | Beaucoup de features categorielles |
| **HistGradientBoosting** (sklearn) | Pas de dep externe, ≈ LightGBM en vitesse | Si on veut rester sklearn-pure |

```python
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

xgb_clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=SEED, eval_metric="logloss")
lgb_clf = lgb.LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=SEED, verbose=-1)
cb_clf  = cb.CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1, random_state=SEED, verbose=0)
```


## 5. Bench complet

In [ ]:
models = {
    "DecisionTree (baseline)": DecisionTreeClassifier(random_state=SEED),
    "Bagging":                 BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=SEED, n_jobs=-1),
    "RandomForest":            RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    "ExtraTrees":              ExtraTreesClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    "AdaBoost":                AdaBoostClassifier(n_estimators=100, random_state=SEED),
    "GradientBoosting":        GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    "HistGradientBoosting":    HistGradientBoostingClassifier(max_iter=100, random_state=SEED),
}

results = []
for name, model in models.items():
    t0 = time.perf_counter()
    score = cross_val_score(model, X_tr, y_tr, cv=5, scoring="f1", n_jobs=-1).mean()
    t = time.perf_counter() - t0
    results.append({"model": name, "cv_f1": score, "time_s": t})

df_b = pd.DataFrame(results).sort_values("cv_f1", ascending=False)
print(df_b.round(4))

## 6. Stacking et Voting — meta-ensembles

In [ ]:
# VotingClassifier : combine les predictions (hard = majoritaire, soft = moyenne proba)
voting = VotingClassifier(estimators=[
    ("lr",  LogisticRegression(max_iter=1000, random_state=SEED)),
    ("rf",  RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)),
    ("gbm", GradientBoostingClassifier(n_estimators=100, random_state=SEED)),
], voting="soft")
voting.fit(X_tr, y_tr)
print(f"VotingClassifier : {voting.score(X_te, y_te):.4f}")

# StackingClassifier : entraine un meta-modele sur les predictions des base models
stack = StackingClassifier(estimators=[
    ("rf",  RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)),
    ("gbm", GradientBoostingClassifier(n_estimators=100, random_state=SEED)),
    ("nb",  GaussianNB()),
], final_estimator=LogisticRegression(max_iter=1000, random_state=SEED), cv=5, n_jobs=-1)
stack.fit(X_tr, y_tr)
print(f"StackingClassifier : {stack.score(X_te, y_te):.4f}")

## 7. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| RF sans `n_estimators >= 100` | RF benefice fortement de + d'arbres (asymptotique) |
| `max_depth=None` sur petit dataset | Overfit fort — limiter max_depth |
| GBM `learning_rate=1.0` | Trop instable — preferer 0.05-0.1 |
| AdaBoost sur dataset bruite | AdaBoost sur-pondere les outliers → GBM mieux |
| Tester boosting sur dataset trivial | Si LogReg fait deja 95%, ensemble = overkill |
| Pas regarder le **temps fit/inference** | Stacking 100x plus lent que LogReg pour 1% de gain |
| RF avec `class_weight=None` sur desequilibre | `class_weight="balanced"` |
| Voting "hard" quand on a les probas | "soft" est presque toujours meilleur |


## References

### Documentation
- sklearn ensemble : https://scikit-learn.org/stable/modules/ensemble.html
- XGBoost : https://xgboost.readthedocs.io/
- LightGBM : https://lightgbm.readthedocs.io/
- CatBoost : https://catboost.ai/docs/

### Voir aussi
- [ML_Regression_Classification_CV_GridSearch.ipynb](ML_Regression_Classification_CV_GridSearch.ipynb)
- [ML_Optimisation_de_Modèles.ipynb](ML_Optimisation_de_Modèles.ipynb)
- [ML_Explication_Feature_Importance_Selection.ipynb](ML_Explication_Feature_Importance_Selection.ipynb)
- [INRIA_SKLearn_MOOC.ipynb](INRIA_SKLearn_MOOC.ipynb)
